In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!nvidia-smi

Tue Sep 22 04:11:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P8             29W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [4]:
!python --version

Python 3.12.13


In [5]:
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --no-cache-dir

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 196.0 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 275.0 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for llama-cpp-python (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for llama-cpp-python
Failed to build llama-cpp-python
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (llama-cpp-python)


In [6]:
!git clone https://github.com/ggml-org/llama.cpp.git

fatal: destination path 'llama.cpp' already exists and is not an empty directory.


In [7]:
%cd llama.cpp

/kaggle/working/llama.cpp


In [8]:
!ls

AGENTS.md	       convert_hf_to_gguf_update.py   pocs
app		       convert_llama_ggml_to_gguf.py  pyproject.toml
AUTHORS		       convert_lora_to_gguf.py	      pyrightconfig.json
benches		       docs			      README.md
build		       examples			      requirements
build-xcframework.sh   flake.nix		      requirements.txt
ci		       ggml			      scripts
CLAUDE.md	       gguf-py			      SECURITY.md
cmake		       grammars			      skills
CMakeLists.txt	       include			      src
CMakePresets.json      LICENSE			      tests
CODEOWNERS	       licenses			      tools
common		       Makefile			      ty.toml
CONTRIBUTING.md        media			      vendor
conversion	       models
convert_hf_to_gguf.py  mypy.ini


In [9]:
!cmake -B build -DGGML_CUDA=ON

-- llama.cpp version: 0.4.1-dev
CMAKE_BUILD_TYPE=Release
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- CUDA Toolkit found
-- Using CMAKE_CUDA_ARCHITECTURES=75 CMAKE_CUDA_ARCHITECTURES_NATIVE=75-real
-- FlashAttention K-V type combinations: f16-f16;q4_0-q4_0;q8_0-q8_0;bf16-bf16
-- CUDA host compiler is GNU 11.4.0
-- Including CUDA backend
-- ggml version: 0.24.0
-- ggml commit:  58367713a
-- OpenSSL found: 3.0.2
-- Generating embedded license file for target: llama-app
-- Configuring done (0.7s)
-- Generating done (0.4s)
-- Build files have been written to: /kaggle/working/llama.cpp/build


In [10]:
!apt-get update -qq
!apt-get install -y libcuda1-550

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package libcuda1-550


In [11]:
!find /usr -name "libcuda.so*" 2>/dev/null

/usr/local/nvidia/lib64/libcuda.so.580.159.04
/usr/local/nvidia/lib64/libcuda.so.1
/usr/local/nvidia/lib64/libcuda.so
/usr/local/cuda-12.8/compat/libcuda.so.1
/usr/local/cuda-12.8/compat/libcuda.so
/usr/local/cuda-12.8/compat/libcuda.so.570.124.06


In [12]:
%cd /kaggle/working/llama.cpp

/kaggle/working/llama.cpp


In [13]:
!rm -rf build

In [14]:
!cmake -B build \
  -DGGML_CUDA=ON \
  -DCMAKE_CUDA_COMPILER=/usr/local/cuda/bin/nvcc \
  -DCMAKE_CUDA_ARCHITECTURES=75 \
  -DCUDAToolkit_ROOT=/usr/local/cuda-12.8 \
  -DCMAKE_LIBRARY_PATH=/usr/local/nvidia/lib64

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 0.4.1-dev
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found Ope

In [15]:
!cmake --build build --config Release -j2

[  1%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml.c.o
[  1%] Building CXX object vendor/hash/CMakeFiles/vendor-hash.dir/hash.cpp.o
[  1%] Building C object vendor/hash/CMakeFiles/vendor-hash.dir/xxhash/xxhash.c.o
[  1%] Building CXX object vendor/hash/CMakeFiles/vendor-hash.dir/sha1/sha1.c.o
[  1%] Building C object vendor/hash/CMakeFiles/vendor-hash.dir/sha256/sha256.c.o
[  2%] Linking CXX static library libvendor-hash.a
[  2%] Built target vendor-hash
[  2%] Building CXX object vendor/cpp-httplib/CMakeFiles/cpp-httplib.dir/httplib.cpp.o
[  2%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml.cpp.o
[  2%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml-alloc.c.o
[  2%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-backend.cpp.o
[  2%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-backend-meta.cpp.o
[  2%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-opt.cpp.o
[  3%] Building CXX object ggml/src/CMakeFiles/ggml-ba

In [16]:
!ls -lh /kaggle/working/llama.cpp/build/bin/llama-cli

-rwxr-xr-x 1 root root 1.4M Sep 22 04:45 /kaggle/working/llama.cpp/build/bin/llama-cli


In [17]:
!/kaggle/working/llama.cpp/build/bin/llama-cli --version

version: 0.4.1-dev (build 11095, commit 58367713a)
built with GNU 11.4.0 for Linux x86_64


In [18]:
!mkdir -p /kaggle/working/models

!pip -q install -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 9.3 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 62.3 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [19]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="cyjin-yl/Qwen3.8-27B-Uncensored-Cyber-agentic-imatrix-GGUF",
    filename="Qwen3.8-27B-Uncensored-Cyber-IQ4_XS-imatrix-fromq8-plus-mtp.gguf",
    local_dir="/kaggle/working/models"
)

print("Model downloaded successfully:")
print(model_path)

Qwen3.8-27B-Uncensored-Cyber-IQ4_XS-imat(…): reconstructing file:   0%|          |  0.00B / 16.5GB            

Qwen3.8-27B-Uncensored-Cyber-IQ4_XS-imat(…): downloading bytes:           |  0.00B            

Model downloaded successfully:
/kaggle/working/models/Qwen3.8-27B-Uncensored-Cyber-IQ4_XS-imatrix-fromq8-plus-mtp.gguf


In [20]:
!/kaggle/working/llama.cpp/build/bin/llama-cli \
-m /kaggle/working/models/Qwen3.8-27B-Uncensored-Cyber-IQ4_XS-imatrix-fromq8-plus-mtp.gguf \
-p "What is machine learning?" \
-n 10000 \
-c 32768 \
--single-turn \
-ngl all



Loading model... 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b11095-58367713a
model      : /kaggle/working/models/Qwen3.8-27B-Uncensored-Cyber-IQ4_XS-imatrix-fromq8-plus-mtp.gguf
ftype      : IQ4_XS - 4.25 bpw
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> What is machine learning?

[Start thinking]

We need answer user: "What is machine learning?" Need produce final. Need likely explain concise but thorough. Include definition, how works, types, examples, pros/cons, when to use, code maybe. Ensure final only answer.
[End thinking]

**Machine learn

In [22]:
!/kaggle/working/llama.cpp/build/bin/llama-cli \
-m /kaggle/working/models/Qwen3.8-27B-Uncensored-Cyber-IQ4_XS-imatrix-fromq8-plus-mtp.gguf \
-p "give me most essential 5 skills name I should develop to build a career in cybersecurity?" \
-n 10000 \
-c 32768 \
--single-turn \
-ngl all



Loading model... 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b11095-58367713a
model      : /kaggle/working/models/Qwen3.8-27B-Uncensored-Cyber-IQ4_XS-imatrix-fromq8-plus-mtp.gguf
ftype      : IQ4_XS - 4.25 bpw
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> give me most essential 5 skills name I should develop to build a career in cybersecurity?

[Start thinking]

The user is asking for the 5 most essential skills to build a career in cybersecurity. They want names. Keep it practical and direct.

Let me think about what truly matters for a cybersecu